In [2]:
from pathlib import Path
import importlib.util
import os
import tempfile

try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False

COLAB_DRIVE_MOUNT = Path("/content/drive")
COLAB_MYDRIVE_ROOT = COLAB_DRIVE_MOUNT / "MyDrive"

PROJECT_ROOT_OVERRIDE = os.environ.get("GRAVITYSPY_PROJECT_ROOT")

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
    if not PROJECT_ROOT.is_absolute():
        raise RuntimeError(
            "GRAVITYSPY_PROJECT_ROOT deve essere un percorso assoluto."
        )
elif IS_COLAB:
    PROJECT_ROOT = COLAB_MYDRIVE_ROOT / "TESI_SAKAI2022"
else:
    raise RuntimeError(
        "Impostare GRAVITYSPY_PROJECT_ROOT su un percorso assoluto "
        "quando il notebook viene eseguito fuori da Colab."
    )

LOCAL_RUNTIME_ROOT_OVERRIDE = os.environ.get(
    "GRAVITYSPY_LOCAL_RUNTIME_ROOT"
)

if LOCAL_RUNTIME_ROOT_OVERRIDE is not None:
    LOCAL_RUNTIME_ROOT = Path(LOCAL_RUNTIME_ROOT_OVERRIDE)
    if not LOCAL_RUNTIME_ROOT.is_absolute():
        raise RuntimeError(
            "GRAVITYSPY_LOCAL_RUNTIME_ROOT deve essere un percorso assoluto."
        )
elif IS_COLAB:
    LOCAL_RUNTIME_ROOT = Path("/content")
else:
    LOCAL_RUNTIME_ROOT = Path(tempfile.gettempdir())

SAKAI224_ROOT = PROJECT_ROOT / "sakai224_full"
TRANSFER_DIR = SAKAI224_ROOT / "transfer_v1"
LOCAL_TRANSFER_DIR = (
    LOCAL_RUNTIME_ROOT / "sakai224_full_transfer_v1"
)
DATA_DIR = LOCAL_RUNTIME_ROOT / "sakai224_full_data"

print("=== NOTEBOOK 01 PREFLIGHT ===")

print(
    "PROJECT_ROOT_EXISTS =",
    PROJECT_ROOT.is_dir(),
)

print(
    "LOCAL_TRANSFER_EXISTS =",
    LOCAL_TRANSFER_DIR.is_dir(),
)

print(
    "LOCAL_DATA_EXISTS =",
    DATA_DIR.is_dir(),
)

=== NOTEBOOK 01 PREFLIGHT ===
DRIVE_MOUNT_EXISTS = False
LOCAL_TRANSFER_EXISTS = False
LOCAL_DATA_EXISTS = False


In [3]:
if IS_COLAB and PROJECT_ROOT_OVERRIDE is None:
    if not COLAB_MYDRIVE_ROOT.is_dir():
        from google.colab import drive

        drive.mount(
            str(COLAB_DRIVE_MOUNT),
            force_remount=False,
        )

Mounted at /content/drive


In [4]:
from pathlib import Path

TRANSFER_DIR = SAKAI224_ROOT / "transfer_v1"

print("TRANSFER_DIR_EXISTS =", TRANSFER_DIR.is_dir())

if TRANSFER_DIR.is_dir():
    print("FILES_FOUND =", len(list(TRANSFER_DIR.iterdir())))

TRANSFER_DIR_EXISTS = True
FILES_FOUND = 7


In [5]:
from pathlib import Path
import shutil

# Staging locale dei sette file transfer_v1 dalla root persistente
# alla cache runtime, senza overwrite della directory di destinazione.
LOCAL_TRANSFER_DIR = (
    LOCAL_RUNTIME_ROOT / "sakai224_full_transfer_v1"
)

if LOCAL_TRANSFER_DIR.exists():
    raise RuntimeError(
        f"STOP: la directory esiste già: {LOCAL_TRANSFER_DIR}"
    )

LOCAL_TRANSFER_DIR.mkdir(parents=True)

print(
    "Staging locale dei 7 file transfer_v1:",
    LOCAL_TRANSFER_DIR,
)

for source in sorted(TRANSFER_DIR.iterdir()):
    if source.is_file():
        destination = LOCAL_TRANSFER_DIR / source.name

        print("File in staging:", source.name)

        shutil.copy2(
            source,
            destination,
        )

# La verifica SHA-256 della copia è eseguita dal blocco dedicato.
print()
print(
    "LOCAL_FILES_FOUND =",
    len(list(LOCAL_TRANSFER_DIR.iterdir())),
)

print(
    "LOCAL_COPY_COMPLETE =",
    len(list(LOCAL_TRANSFER_DIR.iterdir())) == 7,
)

Copio i 7 file da Drive a /content...
Copio: manifest_events_272.csv
Copio: manifest_tensors_224_central.csv
Copio: sakai224_full_central224.tar
Copio: sakai224_full_out272_gray.tar
Copio: split_train_indices_seed123.csv
Copio: split_val_indices_seed123.csv
Copio: transfer_manifest_sha256.txt

LOCAL_FILES_FOUND = 7
LOCAL_COPY_COMPLETE = True


In [6]:
import hashlib

EXPECTED = {
    "sakai224_full_out272_gray.tar":
        "70c766c3652f586c8da4c8cadf134a8ce83cfd07c8cd10545367f380017ddab2",

    "sakai224_full_central224.tar":
        "0fad37cdf4ea056e2c3abfa2d1be73b9d9bd55e644d54a58013950697cf4d4b6",

    "manifest_events_272.csv":
        "620960e6b35da31bc4cf1a8a6da8679027d378e1d4b2ac73c5267c1c5311ae82",

    "manifest_tensors_224_central.csv":
        "94b7e2e051bd32de72dd2cb6a5fb7eda9bcaec123cbf3a07d0d901bd19779afa",

    "split_train_indices_seed123.csv":
        "8c7a513b98dbe996261bc80965500ab15ab77f498ec9440858003152e42f926f",

    "split_val_indices_seed123.csv":
        "e63109c88df7b3ee97317f94791f26df3fcaf8b937921cfb27967d01c2546467",

    "transfer_manifest_sha256.txt":
        "59a04c1098ec4465c1a9864e7bc61181c7981e316c8bc78adbf4c18b78da962f",
}


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(8 * 1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


all_ok = True

print("=== LOCAL SHA-256 CHECK ===")

for name, expected_hash in EXPECTED.items():
    path = LOCAL_TRANSFER_DIR / name
    actual_hash = sha256_file(path)

    ok = actual_hash == expected_hash

    print(name, "=", ok)

    if not ok:
        all_ok = False

print()
print("LOCAL_SHA256_ALL_OK =", all_ok)

=== LOCAL SHA-256 CHECK ===
sakai224_full_out272_gray.tar = True
sakai224_full_central224.tar = True
manifest_events_272.csv = True
manifest_tensors_224_central.csv = True
split_train_indices_seed123.csv = True
split_val_indices_seed123.csv = True
transfer_manifest_sha256.txt = True

LOCAL_SHA256_ALL_OK = True


In [7]:
from pathlib import Path
import subprocess

# Estrazione del TAR out272_gray con guard sulla directory preesistente.
# I gate verificano 34332 PNG e il byte count atteso.
DATA_DIR = LOCAL_RUNTIME_ROOT / "sakai224_full_data"
OUT272_DIR = DATA_DIR / "out272_gray"

if OUT272_DIR.exists():
    raise RuntimeError(
        f"STOP: la directory esiste già: {OUT272_DIR}"
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)

archive = LOCAL_TRANSFER_DIR / "sakai224_full_out272_gray.tar"

print("Estrazione archivio:", archive.name)

subprocess.run(
    [
        "tar",
        "-xf",
        str(archive),
        "-C",
        str(DATA_DIR),
    ],
    check=True,
)

files = [
    path
    for path in OUT272_DIR.rglob("*")
    if path.is_file()
]

total_bytes = sum(
    path.stat().st_size
    for path in files
)

print()
print("EXTRACTION_COMPLETE = True")
print("FILE_COUNT =", len(files))
print("FILE_COUNT_OK =", len(files) == 34332)
print("TOTAL_BYTES =", total_bytes)
print("BYTES_OK =", total_bytes == 1144867540)

Estraggo: sakai224_full_out272_gray.tar

EXTRACTION_COMPLETE = True
FILE_COUNT = 34332
FILE_COUNT_OK = True
TOTAL_BYTES = 1144867540
BYTES_OK = True


In [8]:
# Estrazione del TAR central224 in DATA_DIR con guard sulla directory
# preesistente; i gate verificano 8583 tensori e il byte count atteso.
CENTRAL224_DIR = DATA_DIR / "tensors_224_central_uint8"

if CENTRAL224_DIR.exists():
    raise RuntimeError(
        f"STOP: la directory esiste già: {CENTRAL224_DIR}"
    )

archive = LOCAL_TRANSFER_DIR / "sakai224_full_central224.tar"

print("Estrazione archivio:", archive.name)

subprocess.run(
    [
        "tar",
        "-xf",
        str(archive),
        "-C",
        str(DATA_DIR),
    ],
    check=True,
)

files = [
    path
    for path in CENTRAL224_DIR.rglob("*")
    if path.is_file()
]

total_bytes = sum(
    path.stat().st_size
    for path in files
)

print()
print("EXTRACTION_COMPLETE = True")
print("FILE_COUNT =", len(files))
print("FILE_COUNT_OK =", len(files) == 8583)
print("TOTAL_BYTES =", total_bytes)
print("BYTES_OK =", total_bytes == 1722642432)

Estraggo: sakai224_full_central224.tar

EXTRACTION_COMPLETE = True
FILE_COUNT = 8583
FILE_COUNT_OK = True
TOTAL_BYTES = 1722642432
BYTES_OK = True


In [9]:
import csv

METADATA_DIR = LOCAL_TRANSFER_DIR


def read_csv_dicts(path):
    with path.open(
        "r",
        encoding="utf-8",
        newline=""
    ) as file:
        return list(csv.DictReader(file))


event_rows = read_csv_dicts(
    METADATA_DIR / "manifest_events_272.csv"
)

tensor_rows = read_csv_dicts(
    METADATA_DIR / "manifest_tensors_224_central.csv"
)

train_rows = read_csv_dicts(
    METADATA_DIR / "split_train_indices_seed123.csv"
)

val_rows = read_csv_dicts(
    METADATA_DIR / "split_val_indices_seed123.csv"
)


train_indices = {
    int(row["manifest_index"])
    for row in train_rows
}

val_indices = {
    int(row["manifest_index"])
    for row in val_rows
}


checks = {
    "EVENT_ROWS_OK":
        len(event_rows) == 8583,

    "TENSOR_ROWS_OK":
        len(tensor_rows) == 8583,

    "TRAIN_ROWS_OK":
        len(train_rows) == 6866,

    "VAL_ROWS_OK":
        len(val_rows) == 1717,

    "TRAIN_UNIQUE_OK":
        len(train_indices) == 6866,

    "VAL_UNIQUE_OK":
        len(val_indices) == 1717,

    "OVERLAP_ZERO":
        len(train_indices & val_indices) == 0,

    "FULL_COVERAGE":
        train_indices | val_indices == set(range(8583)),
}


print("=== NOTEBOOK 01 DATA CONTRACT ===")

for name, value in checks.items():
    print(name, "=", value)

print()
print(
    "DATA_CONTRACT_PASS =",
    all(checks.values()),
)

=== NOTEBOOK 01 DATA CONTRACT ===
EVENT_ROWS_OK = True
TENSOR_ROWS_OK = True
TRAIN_ROWS_OK = True
VAL_ROWS_OK = True
TRAIN_UNIQUE_OK = True
VAL_UNIQUE_OK = True
OVERLAP_ZERO = True
FULL_COVERAGE = True

DATA_CONTRACT_PASS = True


In [10]:
MASK64 = (1 << 64) - 1

GS_AUGMENT_MODE_LEGACY_TEXT = 0
GS_AUGMENT_MODE_SYMMETRIC_FIGURE = 1

GS_AUGMENT_STATELESS_V1_DOMAIN_TAG = 0x4753415547563121


def c_u64(value):
    return value & MASK64


def gs_augment_stateless_mix64(value):
    value = c_u64(value + 0x9E3779B97F4A7C15)

    value = c_u64(
        (value ^ (value >> 30))
        * 0xBF58476D1CE4E5B9
    )

    value = c_u64(
        (value ^ (value >> 27))
        * 0x94D049BB133111EB
    )

    return c_u64(value ^ (value >> 31))


def gs_augment_stateless_value(
    base_seed,
    epoch,
    manifest_index,
    mode,
    stream_index,
):
    h = gs_augment_stateless_mix64(
        GS_AUGMENT_STATELESS_V1_DOMAIN_TAG
    )

    h = gs_augment_stateless_mix64(h ^ c_u64(base_seed))
    h = gs_augment_stateless_mix64(h ^ c_u64(epoch))
    h = gs_augment_stateless_mix64(h ^ c_u64(manifest_index))
    h = gs_augment_stateless_mix64(h ^ c_u64(mode))

    return gs_augment_stateless_mix64(
        h ^ c_u64(stream_index)
    )


def gs_augment_crop_left_for_event_mode(
    base_seed,
    epoch,
    manifest_index,
    mode,
):
    if mode == GS_AUGMENT_MODE_LEGACY_TEXT:
        min_crop = 0
        max_crop = 24

    elif mode == GS_AUGMENT_MODE_SYMMETRIC_FIGURE:
        min_crop = 0
        max_crop = 48

    else:
        raise ValueError("mode non valido")

    crop_range = max_crop - min_crop + 1

    threshold = c_u64(
        0 - crop_range
    ) % crop_range

    stream_index = 0

    while True:
        value = gs_augment_stateless_value(
            base_seed,
            epoch,
            manifest_index,
            mode,
            stream_index,
        )

        if value >= threshold:
            return min_crop + value % crop_range

        stream_index += 1


# Golden vectors del test C reale
golden_vectors = [
    (0, 0, 0, 0, 7),
    (123, 0, 7663, 1, 36),
    (789, 79, 4224, 1, 25),
    (456, 3, 42, 0, 0),
    (MASK64, MASK64, MASK64, 1, 40),
]


print("=== AUGMENTATION C GOLDEN VECTORS ===")

all_ok = True

for seed, epoch, manifest_index, mode, expected in golden_vectors:
    actual = gs_augment_crop_left_for_event_mode(
        seed,
        epoch,
        manifest_index,
        mode,
    )

    ok = actual == expected

    print(
        "expected =", expected,
        "actual =", actual,
        "OK =", ok,
    )

    if not ok:
        all_ok = False

print()
print("AUGMENTATION_GOLDEN_PASS =", all_ok)

=== AUGMENTATION C GOLDEN VECTORS ===
expected = 7 actual = 7 OK = True
expected = 36 actual = 36 OK = True
expected = 25 actual = 25 OK = True
expected = 0 actual = 0 OK = True
expected = 40 actual = 40 OK = True

AUGMENTATION_GOLDEN_PASS = True


In [11]:
train_manifest_indices = [
    int(row["manifest_index"])
    for row in train_rows
]

AUGMENTATION_SEED = 789
MODE = GS_AUGMENT_MODE_SYMMETRIC_FIGURE


def make_crop_map(indices, epoch):
    return {
        manifest_index:
        gs_augment_crop_left_for_event_mode(
            AUGMENTATION_SEED,
            epoch,
            manifest_index,
            MODE,
        )
        for manifest_index in indices
    }


# Ordine originale
map_original = make_crop_map(
    train_manifest_indices,
    epoch=0,
)

# Ordine invertito
map_reversed = make_crop_map(
    reversed(train_manifest_indices),
    epoch=0,
)

# Stessi eventi ma divisi in batch artificiali da 37
map_chunked = {}

batch_size_test = 37

for start in range(
    0,
    len(train_manifest_indices),
    batch_size_test,
):
    chunk = train_manifest_indices[
        start:start + batch_size_test
    ]

    map_chunked.update(
        make_crop_map(chunk, epoch=0)
    )


# Epoch successiva
map_epoch1 = make_crop_map(
    train_manifest_indices,
    epoch=1,
)

values_epoch0 = list(map_original.values())

changed_between_epochs = sum(
    map_original[index] != map_epoch1[index]
    for index in train_manifest_indices
)


print("=== FULL TRAIN STATELESS TEST ===")

print(
    "ORDER_INDEPENDENT =",
    map_original == map_reversed,
)

print(
    "BATCH_BOUNDARY_INDEPENDENT =",
    map_original == map_chunked,
)

print(
    "ALL_IN_0_48 =",
    all(0 <= value <= 48 for value in values_epoch0),
)

print(
    "DISTINCT_CROPS =",
    len(set(values_epoch0)),
)

print(
    "MIN_CROP =",
    min(values_epoch0),
)

print(
    "MAX_CROP =",
    max(values_epoch0),
)

print(
    "EVENTS_CHANGED_EPOCH0_TO_1 =",
    changed_between_epochs,
)

print()
print(
    "STATELESS_FULL_TRAIN_PASS =",
    (
        map_original == map_reversed
        and map_original == map_chunked
        and all(0 <= value <= 48 for value in values_epoch0)
        and changed_between_epochs > 0
    ),
)

=== FULL TRAIN STATELESS TEST ===
ORDER_INDEPENDENT = True
BATCH_BOUNDARY_INDEPENDENT = True
ALL_IN_0_48 = True
DISTINCT_CROPS = 49
MIN_CROP = 0
MAX_CROP = 48
EVENTS_CHANGED_EPOCH0_TO_1 = 6715

STATELESS_FULL_TRAIN_PASS = True


In [12]:
from PIL import Image
import numpy as np
import torch


VIEW_COLUMNS = [
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]


def load_train_event_224(
    manifest_index,
    epoch,
    augmentation_seed=789,
):
    """
    Train PROJECT:
    manifest_index + epoch
    -> crop stateless symmetric_figure
    -> 4 PNG grayscale 272x224
    -> stesso crop sui 4 canali
    -> torch.float32 (4,224,224) in [0,1]
    """

    event = event_rows[manifest_index]

    crop_left = gs_augment_crop_left_for_event_mode(
        augmentation_seed,
        epoch,
        manifest_index,
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
    )

    channels = []

    for column in VIEW_COLUMNS:
        path = DATA_DIR / event[column]

        with Image.open(path) as image:
            if image.mode != "L":
                raise RuntimeError(
                    f"PNG non grayscale: {path}"
                )

            array = np.asarray(
                image,
                dtype=np.uint8,
            ).copy()

        if array.shape != (224, 272):
            raise RuntimeError(
                f"Shape PNG errata: {array.shape}"
            )

        crop = array[
            :,
            crop_left:crop_left + 224
        ]

        channels.append(crop)

    tensor_u8 = np.stack(
        channels,
        axis=0,
    )

    tensor = torch.from_numpy(
        tensor_u8.astype(np.float32)
        / np.float32(255.0)
    )

    return tensor, crop_left, event["event_id"]


print("=== REAL TRAIN LOADER TEST ===")

loader_ok = True

for row in train_rows[:4]:
    manifest_index = int(row["manifest_index"])

    tensor, crop_left, event_id = load_train_event_224(
        manifest_index,
        epoch=0,
    )

    checks = (
        tuple(tensor.shape) == (4, 224, 224)
        and tensor.dtype == torch.float32
        and torch.isfinite(tensor).all().item()
        and tensor.min().item() >= 0.0
        and tensor.max().item() <= 1.0
        and 0 <= crop_left <= 48
        and event_id == row["event_id"]
    )

    print(
        "manifest_index =", manifest_index,
        "event_id =", event_id,
        "crop_left =", crop_left,
        "OK =", checks,
    )

    if not checks:
        loader_ok = False


print()
print(
    "FIRST_EVENT_CROP_EXPECTED_36 =",
    load_train_event_224(
        int(train_rows[0]["manifest_index"]),
        epoch=0,
    )[1] == 36,
)

print(
    "REAL_TRAIN_LOADER_PASS =",
    loader_ok,
)

=== REAL TRAIN LOADER TEST ===
manifest_index = 7663 event_id = H1_BqniZ6oLFl crop_left = 36 OK = True
manifest_index = 2606 event_id = H1_AYvIQZztiZ crop_left = 20 OK = True
manifest_index = 7841 event_id = H1_j34emtOH9H crop_left = 33 OK = True
manifest_index = 4067 event_id = L1_uEOTA8sCmI crop_left = 5 OK = True

FIRST_EVENT_CROP_EXPECTED_36 = True
REAL_TRAIN_LOADER_PASS = True


In [13]:
def load_validation_event_224(manifest_index):
    """
    Validation PROJECT:
    manifest_index
    -> .bin centrale ufficiale
    -> uint8 (4,224,224), channel-major
    -> torch.float32 [0,1]
    """

    tensor_row = tensor_rows[manifest_index]
    path = DATA_DIR / tensor_row["tensor_path"]

    if not path.is_file():
        raise RuntimeError(
            f"Tensore mancante: {path}"
        )

    tensor_u8 = np.fromfile(
        path,
        dtype=np.uint8,
    )

    expected_values = 4 * 224 * 224

    if tensor_u8.size != expected_values:
        raise RuntimeError(
            f"Numero valori errato: {tensor_u8.size}"
        )

    tensor_u8 = tensor_u8.reshape(
        4,
        224,
        224,
    )

    tensor = torch.from_numpy(
        tensor_u8.astype(np.float32)
        / np.float32(255.0)
    )

    return (
        tensor,
        tensor_row["event_id"],
        int(tensor_row["crop_left"]),
    )


print("=== REAL VALIDATION LOADER TEST ===")

loader_ok = True

for row in val_rows[:4]:
    manifest_index = int(row["manifest_index"])

    tensor, event_id, crop_left = load_validation_event_224(
        manifest_index
    )

    checks = (
        tuple(tensor.shape) == (4, 224, 224)
        and tensor.dtype == torch.float32
        and torch.isfinite(tensor).all().item()
        and tensor.min().item() >= 0.0
        and tensor.max().item() <= 1.0
        and crop_left == 24
        and event_id == row["event_id"]
    )

    print(
        "manifest_index =", manifest_index,
        "event_id =", event_id,
        "crop_left =", crop_left,
        "OK =", checks,
    )

    if not checks:
        loader_ok = False


print()
print(
    "REAL_VALIDATION_LOADER_PASS =",
    loader_ok,
)

=== REAL VALIDATION LOADER TEST ===
manifest_index = 6864 event_id = H1_4D7IcowE8a crop_left = 24 OK = True
manifest_index = 1488 event_id = H1_NGodo27RM7 crop_left = 24 OK = True
manifest_index = 8384 event_id = L1_FgsynKioon crop_left = 24 OK = True
manifest_index = 6585 event_id = H1_baOcw3xyeE crop_left = 24 OK = True

REAL_VALIDATION_LOADER_PASS = True


In [14]:
def load_central_from_png_u8(manifest_index):
    event = event_rows[manifest_index]

    channels = []

    for column in VIEW_COLUMNS:
        path = DATA_DIR / event[column]

        with Image.open(path) as image:
            array = np.asarray(
                image,
                dtype=np.uint8,
            ).copy()

        channels.append(
            array[:, 24:24 + 224]
        )

    return np.stack(channels, axis=0)


def load_central_bin_u8(manifest_index):
    tensor_row = tensor_rows[manifest_index]
    path = DATA_DIR / tensor_row["tensor_path"]

    return np.fromfile(
        path,
        dtype=np.uint8,
    ).reshape(4, 224, 224)


test_indices = [
    int(train_rows[0]["manifest_index"]),
    int(val_rows[0]["manifest_index"]),
]

print("=== CENTRAL PNG vs BIN AUDIT ===")

all_ok = True

for manifest_index in test_indices:
    from_png = load_central_from_png_u8(
        manifest_index
    )

    from_bin = load_central_bin_u8(
        manifest_index
    )

    match = np.array_equal(
        from_png,
        from_bin,
    )

    print(
        "manifest_index =", manifest_index,
        "event_id =", event_rows[manifest_index]["event_id"],
        "BITWISE_MATCH =", match,
    )

    if not match:
        all_ok = False

print()
print(
    "CENTRAL_PNG_BIN_AUDIT_PASS =",
    all_ok,
)

=== CENTRAL PNG vs BIN AUDIT ===
manifest_index = 7663 event_id = H1_BqniZ6oLFl BITWISE_MATCH = True
manifest_index = 6864 event_id = H1_4D7IcowE8a BITWISE_MATCH = True

CENTRAL_PNG_BIN_AUDIT_PASS = True


In [15]:
from torch.utils.data import Dataset


class GravitySpy224Dataset(Dataset):
    """
    Dataset PROJECT per sakai224_full.

    TRAIN:
        split train
        -> PNG 272x224
        -> crop stateless dipendente da epoch e manifest_index

    VALIDATION:
        split validation
        -> tensore centrale .bin
        -> nessuna augmentation
    """

    def __init__(
        self,
        split_rows,
        split_name,
        epoch=0,
        augmentation_seed=789,
    ):
        if split_name not in ("train", "validation"):
            raise ValueError(
                "split_name deve essere 'train' o 'validation'"
            )

        self.split_rows = split_rows
        self.split_name = split_name
        self.epoch = int(epoch)
        self.augmentation_seed = int(augmentation_seed)

    def __len__(self):
        return len(self.split_rows)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __getitem__(self, index):
        row = self.split_rows[index]
        manifest_index = int(row["manifest_index"])

        if self.split_name == "train":
            tensor, crop_left, event_id = load_train_event_224(
                manifest_index,
                epoch=self.epoch,
                augmentation_seed=self.augmentation_seed,
            )

        else:
            tensor, event_id, crop_left = load_validation_event_224(
                manifest_index
            )

        return {
            "x": tensor,
            "manifest_index": manifest_index,
            "event_id": event_id,
            "crop_left": crop_left,
        }


train_dataset = GravitySpy224Dataset(
    train_rows,
    split_name="train",
    epoch=0,
)

val_dataset = GravitySpy224Dataset(
    val_rows,
    split_name="validation",
)


print("=== DATASET CREATED ===")
print("TRAIN_LEN =", len(train_dataset))
print("VAL_LEN   =", len(val_dataset))

=== DATASET CREATED ===
TRAIN_LEN = 6866
VAL_LEN   = 1717


In [16]:
train_item = train_dataset[0]
val_item = val_dataset[0]

print("=== TRAIN ITEM ===")
print("shape          =", tuple(train_item["x"].shape))
print("dtype          =", train_item["x"].dtype)
print("manifest_index =", train_item["manifest_index"])
print("event_id       =", train_item["event_id"])
print("crop_left      =", train_item["crop_left"])

print()

print("=== VALIDATION ITEM ===")
print("shape          =", tuple(val_item["x"].shape))
print("dtype          =", val_item["x"].dtype)
print("manifest_index =", val_item["manifest_index"])
print("event_id       =", val_item["event_id"])
print("crop_left      =", val_item["crop_left"])

print()

print(
    "DATASET_ITEM_PASS =",
    (
        tuple(train_item["x"].shape) == (4, 224, 224)
        and tuple(val_item["x"].shape) == (4, 224, 224)
        and train_item["x"].dtype == torch.float32
        and val_item["x"].dtype == torch.float32
        and train_item["manifest_index"] == 7663
        and train_item["crop_left"] == 36
        and val_item["manifest_index"] == 6864
        and val_item["crop_left"] == 24
    ),
)

=== TRAIN ITEM ===
shape          = (4, 224, 224)
dtype          = torch.float32
manifest_index = 7663
event_id       = H1_BqniZ6oLFl
crop_left      = 36

=== VALIDATION ITEM ===
shape          = (4, 224, 224)
dtype          = torch.float32
manifest_index = 6864
event_id       = H1_4D7IcowE8a
crop_left      = 24

DATASET_ITEM_PASS = True


In [17]:
from torch.utils.data import DataLoader

train_loader_test = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)

val_loader_test = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)

train_batch = next(iter(train_loader_test))
val_batch = next(iter(val_loader_test))

print("=== TRAIN BATCH ===")
print("x shape =", tuple(train_batch["x"].shape))
print("x dtype =", train_batch["x"].dtype)
print("manifest_index =", train_batch["manifest_index"].tolist())
print("crop_left =", train_batch["crop_left"].tolist())

print()

print("=== VALIDATION BATCH ===")
print("x shape =", tuple(val_batch["x"].shape))
print("x dtype =", val_batch["x"].dtype)
print("manifest_index =", val_batch["manifest_index"].tolist())
print("crop_left =", val_batch["crop_left"].tolist())

print()

batch_pass = (
    tuple(train_batch["x"].shape) == (4, 4, 224, 224)
    and tuple(val_batch["x"].shape) == (4, 4, 224, 224)
    and train_batch["x"].dtype == torch.float32
    and val_batch["x"].dtype == torch.float32
    and train_batch["manifest_index"].tolist()
        == [int(row["manifest_index"]) for row in train_rows[:4]]
    and val_batch["manifest_index"].tolist()
        == [int(row["manifest_index"]) for row in val_rows[:4]]
    and val_batch["crop_left"].tolist() == [24, 24, 24, 24]
)

print("DATALOADER_BATCH_PASS =", batch_pass)

=== TRAIN BATCH ===
x shape = (4, 4, 224, 224)
x dtype = torch.float32
manifest_index = [7663, 2606, 7841, 4067]
crop_left = [36, 20, 33, 5]

=== VALIDATION BATCH ===
x shape = (4, 4, 224, 224)
x dtype = torch.float32
manifest_index = [6864, 1488, 8384, 6585]
crop_left = [24, 24, 24, 24]

DATALOADER_BATCH_PASS = True


In [18]:
# ============================================================
# REPLICA PYTHON ESATTA DELLO SHUFFLE PROJECT C
# ============================================================

MASK64 = (1 << 64) - 1

VAE_C_RNG_DOMAIN_SHUFFLE = 2

SPLITMIX_INCREMENT = 0x9E3779B97F4A7C15
SPLITMIX_MUL1 = 0xBF58476D1CE4E5B9
SPLITMIX_MUL2 = 0x94D049BB133111EB

DERIVE_SEED_TAG = 0xD1B54A32D192ED03


def u64(value):
    """Emula l'aritmetica uint64_t del C."""
    return value & MASK64


def vae_c_rng_mix64_py(value):
    """
    Replica esatta di vae_c_rng_mix64().
    """
    value = u64(
        value + SPLITMIX_INCREMENT
    )

    value = u64(
        (value ^ (value >> 30))
        * SPLITMIX_MUL1
    )

    value = u64(
        (value ^ (value >> 27))
        * SPLITMIX_MUL2
    )

    return u64(
        value ^ (value >> 31)
    )


def vae_c_rng_derive_seed_py(
    base_seed,
    domain,
    epoch,
    manifest_index,
    stream_index,
):
    """
    Replica esatta di vae_c_rng_derive_seed().
    """

    h = vae_c_rng_mix64_py(
        u64(base_seed) ^ DERIVE_SEED_TAG
    )

    h = vae_c_rng_mix64_py(
        h ^ u64(domain)
    )

    h = vae_c_rng_mix64_py(
        h ^ u64(epoch)
    )

    h = vae_c_rng_mix64_py(
        h ^ u64(manifest_index)
    )

    h = vae_c_rng_mix64_py(
        h ^ u64(stream_index)
    )

    return h


def vae_c_rng_next_u64_py(state):
    """
    Replica esatta di vae_c_rng_next_u64().

    Ritorna:
        nuovo_state, valore_random
    """

    state = u64(
        state + SPLITMIX_INCREMENT
    )

    value = state

    value = u64(
        (value ^ (value >> 30))
        * SPLITMIX_MUL1
    )

    value = u64(
        (value ^ (value >> 27))
        * SPLITMIX_MUL2
    )

    value = u64(
        value ^ (value >> 31)
    )

    return state, value


def vae_c_shuffle_bounded_u64_py(
    state,
    upper_bound_exclusive,
):
    """
    Replica di vae_c_shuffle_bounded_u64():
    rejection sampling senza modulo bias.
    """

    if upper_bound_exclusive <= 0:
        raise ValueError(
            "upper_bound_exclusive deve essere > 0"
        )

    threshold = (
        u64(0 - upper_bound_exclusive)
        % upper_bound_exclusive
    )

    while True:
        state, random_value = vae_c_rng_next_u64_py(
            state
        )

        if random_value >= threshold:
            return (
                state,
                random_value % upper_bound_exclusive,
            )


def vae_c_shuffle_make_permutation_py(
    count,
    base_seed,
    epoch,
):
    """
    Replica di vae_c_shuffle_make_permutation().
    """

    if count < 0:
        raise ValueError("count non valido")

    if count == 0:
        return []

    derived_seed = vae_c_rng_derive_seed_py(
        base_seed,
        VAE_C_RNG_DOMAIN_SHUFFLE,
        epoch,
        0,
        0,
    )

    state = derived_seed

    permutation = list(range(count))

    # Fisher-Yates identico al C:
    # index = count-1 ... 1
    for index in range(
        count - 1,
        0,
        -1,
    ):
        state, selected = (
            vae_c_shuffle_bounded_u64_py(
                state,
                index + 1,
            )
        )

        permutation[index], permutation[selected] = (
            permutation[selected],
            permutation[index],
        )

    return permutation


# ============================================================
# GOLDEN VECTORS DAL TEST C REALE
# ============================================================

golden_cases = [
    (
        8,
        101112,
        0,
        [0, 5, 4, 7, 1, 3, 6, 2],
    ),
    (
        8,
        101112,
        1,
        [2, 3, 1, 4, 0, 5, 6, 7],
    ),
    (
        8,
        202224,
        0,
        [3, 6, 4, 2, 1, 7, 5, 0],
    ),
    (
        12,
        101112,
        7,
        [8, 11, 10, 6, 2, 0, 9, 4, 5, 1, 7, 3],
    ),
]


print("=== C <-> PYTHON SHUFFLE GOLDEN VECTORS ===")

all_golden_ok = True

for count, seed, epoch, expected in golden_cases:

    actual = vae_c_shuffle_make_permutation_py(
        count,
        seed,
        epoch,
    )

    ok = actual == expected

    print()
    print("count    =", count)
    print("seed     =", seed)
    print("epoch    =", epoch)
    print("expected =", expected)
    print("actual   =", actual)
    print("OK       =", ok)

    if not ok:
        all_golden_ok = False


# ============================================================
# REJECTION PATH DEL TEST C
# ============================================================

state_before_zero_output = 0x61C8864680B583EB

final_state, rejection_value = (
    vae_c_shuffle_bounded_u64_py(
        state_before_zero_output,
        10,
    )
)

rejection_path_ok = (
    final_state == SPLITMIX_INCREMENT
    and rejection_value == 5
)


print()
print(
    "SHUFFLE_GOLDEN_VECTORS_PASS =",
    all_golden_ok,
)

print(
    "REJECTION_PATH_PASS =",
    rejection_path_ok,
)

print(
    "SHUFFLE_C_PYTHON_PASS =",
    all_golden_ok and rejection_path_ok,
)

=== C <-> PYTHON SHUFFLE GOLDEN VECTORS ===

count    = 8
seed     = 101112
epoch    = 0
expected = [0, 5, 4, 7, 1, 3, 6, 2]
actual   = [0, 5, 4, 7, 1, 3, 6, 2]
OK       = True

count    = 8
seed     = 101112
epoch    = 1
expected = [2, 3, 1, 4, 0, 5, 6, 7]
actual   = [2, 3, 1, 4, 0, 5, 6, 7]
OK       = True

count    = 8
seed     = 202224
epoch    = 0
expected = [3, 6, 4, 2, 1, 7, 5, 0]
actual   = [3, 6, 4, 2, 1, 7, 5, 0]
OK       = True

count    = 12
seed     = 101112
epoch    = 7
expected = [8, 11, 10, 6, 2, 0, 9, 4, 5, 1, 7, 3]
actual   = [8, 11, 10, 6, 2, 0, 9, 4, 5, 1, 7, 3]
OK       = True

SHUFFLE_GOLDEN_VECTORS_PASS = True
REJECTION_PATH_PASS = True
SHUFFLE_C_PYTHON_PASS = True


In [19]:
SHUFFLE_BASE_SEED = 101112
TRAIN_COUNT = len(train_dataset)

perm_epoch0_a = vae_c_shuffle_make_permutation_py(
    TRAIN_COUNT,
    SHUFFLE_BASE_SEED,
    0,
)

perm_epoch0_b = vae_c_shuffle_make_permutation_py(
    TRAIN_COUNT,
    SHUFFLE_BASE_SEED,
    0,
)

perm_epoch1 = vae_c_shuffle_make_permutation_py(
    TRAIN_COUNT,
    SHUFFLE_BASE_SEED,
    1,
)


def permutation_is_valid(permutation, count):
    return (
        len(permutation) == count
        and set(permutation) == set(range(count))
    )


source_positions = list(range(TRAIN_COUNT))

source_unchanged_before = source_positions.copy()

reordered_manifest_epoch0 = [
    int(train_rows[position]["manifest_index"])
    for position in perm_epoch0_a
]


checks = {
    "COUNT_OK":
        len(perm_epoch0_a) == 6866,

    "PERMUTATION_VALID":
        permutation_is_valid(
            perm_epoch0_a,
            TRAIN_COUNT,
        ),

    "REPEATABLE_SAME_EPOCH":
        perm_epoch0_a == perm_epoch0_b,

    "EPOCH_CHANGES_ORDER":
        perm_epoch0_a != perm_epoch1,

    "SOURCE_POSITIONS_UNCHANGED":
        source_positions == source_unchanged_before,

    "REORDERED_MANIFEST_UNIQUE":
        len(set(reordered_manifest_epoch0)) == 6866,

    "REORDERED_MANIFEST_SAME_SET":
        set(reordered_manifest_epoch0)
        == {
            int(row["manifest_index"])
            for row in train_rows
        },
}


print("=== FULL TRAIN SHUFFLE AUDIT ===")

for name, value in checks.items():
    print(name, "=", value)

print()
print(
    "FIRST_12_PERM_POS_EPOCH0 =",
    perm_epoch0_a[:12],
)

print(
    "FIRST_12_MANIFEST_EPOCH0 =",
    reordered_manifest_epoch0[:12],
)

print()
print(
    "FULL_TRAIN_SHUFFLE_PASS =",
    all(checks.values()),
)

=== FULL TRAIN SHUFFLE AUDIT ===
COUNT_OK = True
PERMUTATION_VALID = True
REPEATABLE_SAME_EPOCH = True
EPOCH_CHANGES_ORDER = True
SOURCE_POSITIONS_UNCHANGED = True
REORDERED_MANIFEST_UNIQUE = True
REORDERED_MANIFEST_SAME_SET = True

FIRST_12_PERM_POS_EPOCH0 = [752, 2659, 4819, 4649, 4369, 3945, 4482, 2825, 2403, 3407, 1964, 286]
FIRST_12_MANIFEST_EPOCH0 = [2676, 6250, 2396, 5602, 2168, 1732, 5987, 1103, 5829, 596, 4586, 847]

FULL_TRAIN_SHUFFLE_PASS = True


In [20]:
from torch.utils.data import (
    Sampler,
    DataLoader,
    BatchSampler,
)


class ProjectEpochSampler(Sampler):
    """
    Sampler PyTorch equivalente alla semantica del runner C.

    Produce una permutazione delle POSIZIONI dello split:
        0 .. count-1

    Il Dataset traduce poi ogni posizione nel manifest_index
    presente nella corrispondente riga dello split.
    """

    def __init__(
        self,
        count,
        base_seed=101112,
        epoch=0,
    ):
        self.count = int(count)
        self.base_seed = int(base_seed)
        self.epoch = int(epoch)

        if self.count < 0:
            raise ValueError("count non valido")

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        permutation = vae_c_shuffle_make_permutation_py(
            self.count,
            self.base_seed,
            self.epoch,
        )

        return iter(permutation)

    def __len__(self):
        return self.count


train_sampler = ProjectEpochSampler(
    count=len(train_dataset),
    base_seed=101112,
    epoch=0,
)


def set_project_train_epoch(epoch):
    """
    Lo stesso epoch controlla:
      1. shuffle delle posizioni;
      2. augmentation stateless per manifest_index.
    """
    epoch = int(epoch)

    train_sampler.set_epoch(epoch)
    train_dataset.set_epoch(epoch)


train_loader_project = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=train_sampler,
    num_workers=0,
    drop_last=False,
)

val_loader_project = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)


print("=== PROJECT SAMPLER CREATED ===")
print("TRAIN_SAMPLES =", len(train_sampler))
print("TRAIN_BATCHES =", len(train_loader_project))
print("VAL_BATCHES   =", len(val_loader_project))

=== PROJECT SAMPLER CREATED ===
TRAIN_SAMPLES = 6866
TRAIN_BATCHES = 1717
VAL_BATCHES   = 430


### Nota metodologica: label barrier e determinismo

I manifest canonici includono il campo `label` e `csv.DictReader` lo materializza nelle righe lette:

- `LABELS_PRESENT_AS_METADATA = YES`
- `LABELS_PARSED_IN_MANIFEST_ROWS = YES`
- `LABELS_USED_FOR_COMPUTATION = NO`

Le label non partecipano a split, augmentation, shuffle, Dataset, sampler, costruzione dei batch o controlli numerici del loader.

Shuffle e augmentation sono deterministici, usano seed e domini distinti e condividono l'epoch come input senza condividere uno stato RNG mutabile. Lo shuffle permuta le posizioni dello split; l'augmentation dipende dall'identità dell'evento tramite `manifest_index`. Il crop di un evento è quindi indipendente dalla posizione nel batch e dall'ordine di iterazione.

In [21]:
# Verifica integrata di ProjectEpochSampler e Dataset train:
# propagazione dell'epoch, identità manifest, crop stateless,
# geometria dei batch e consumo completo dello split.
# ============================================================
# EPOCH 0
# ============================================================

set_project_train_epoch(0)

expected_permutation_epoch0 = (
    vae_c_shuffle_make_permutation_py(
        len(train_dataset),
        101112,
        0,
    )
)

expected_positions_epoch0 = (
    expected_permutation_epoch0[:4]
)

expected_manifest_epoch0 = [
    int(train_rows[position]["manifest_index"])
    for position in expected_positions_epoch0
]

expected_crops_epoch0 = [
    gs_augment_crop_left_for_event_mode(
        789,
        0,
        manifest_index,
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
    )
    for manifest_index in expected_manifest_epoch0
]


train_batch_epoch0 = next(
    iter(train_loader_project)
)

actual_manifest_epoch0 = (
    train_batch_epoch0["manifest_index"].tolist()
)

actual_crops_epoch0 = (
    train_batch_epoch0["crop_left"].tolist()
)


# Costruzione dei confini di batch usando solamente gli indici:
# questa operazione non legge le immagini.
index_batches_epoch0 = list(
    BatchSampler(
        train_sampler,
        batch_size=4,
        drop_last=False,
    )
)


# ============================================================
# EPOCH 1
# ============================================================

set_project_train_epoch(1)

train_batch_epoch1 = next(
    iter(train_loader_project)
)

actual_manifest_epoch1 = (
    train_batch_epoch1["manifest_index"].tolist()
)


# Ripristino di epoch 0 come stato finale del test.
set_project_train_epoch(0)


# ============================================================
# HARD GATE
# ============================================================

checks = {
    "FIRST_POSITIONS_MATCH_FROZEN_AUDIT":
        expected_positions_epoch0
        == [752, 2659, 4819, 4649],

    "FIRST_MANIFEST_MATCH_EXPECTED":
        actual_manifest_epoch0
        == expected_manifest_epoch0,

    "FIRST_CROPS_MATCH_EXPECTED":
        actual_crops_epoch0
        == expected_crops_epoch0,

    "TRAIN_BATCH_SHAPE_OK":
        tuple(train_batch_epoch0["x"].shape)
        == (4, 4, 224, 224),

    "TRAIN_BATCH_DTYPE_OK":
        train_batch_epoch0["x"].dtype
        == torch.float32,

    "BATCH_COUNT_OK":
        len(index_batches_epoch0) == 1717,

    "TAIL_BATCH_SIZE_OK":
        len(index_batches_epoch0[-1]) == 2,

    "ALL_POSITIONS_CONSUMED_ONCE":
        sorted(
            position
            for batch in index_batches_epoch0
            for position in batch
        )
        == list(range(6866)),

    "EPOCH_CHANGES_FIRST_BATCH":
        actual_manifest_epoch0
        != actual_manifest_epoch1,

    "SOURCE_SPLIT_UNCHANGED":
        [
            int(row["manifest_index"])
            for row in train_rows[:4]
        ]
        == [7663, 2606, 7841, 4067],
}


print("=== PROJECT SAMPLER INTEGRATION AUDIT ===")

print(
    "EXPECTED_POSITIONS_EPOCH0 =",
    expected_positions_epoch0,
)

print(
    "EXPECTED_MANIFEST_EPOCH0 =",
    expected_manifest_epoch0,
)

print(
    "ACTUAL_MANIFEST_EPOCH0   =",
    actual_manifest_epoch0,
)

print(
    "EXPECTED_CROPS_EPOCH0 =",
    expected_crops_epoch0,
)

print(
    "ACTUAL_CROPS_EPOCH0   =",
    actual_crops_epoch0,
)

print(
    "ACTUAL_MANIFEST_EPOCH1 =",
    actual_manifest_epoch1,
)

print()

for name, value in checks.items():
    print(name, "=", value)

print()
print(
    "PROJECT_SAMPLER_INTEGRATION_PASS =",
    all(checks.values()),
)

=== PROJECT SAMPLER INTEGRATION AUDIT ===
EXPECTED_POSITIONS_EPOCH0 = [752, 2659, 4819, 4649]
EXPECTED_MANIFEST_EPOCH0 = [2676, 6250, 2396, 5602]
ACTUAL_MANIFEST_EPOCH0   = [2676, 6250, 2396, 5602]
EXPECTED_CROPS_EPOCH0 = [20, 15, 19, 12]
ACTUAL_CROPS_EPOCH0   = [20, 15, 19, 12]
ACTUAL_MANIFEST_EPOCH1 = [6028, 4009, 12, 5170]

FIRST_POSITIONS_MATCH_FROZEN_AUDIT = True
FIRST_MANIFEST_MATCH_EXPECTED = True
FIRST_CROPS_MATCH_EXPECTED = True
TRAIN_BATCH_SHAPE_OK = True
TRAIN_BATCH_DTYPE_OK = True
BATCH_COUNT_OK = True
TAIL_BATCH_SIZE_OK = True
ALL_POSITIONS_CONSUMED_ONCE = True
EPOCH_CHANGES_FIRST_BATCH = True
SOURCE_SPLIT_UNCHANGED = True

PROJECT_SAMPLER_INTEGRATION_PASS = True
